In [1]:
from google.colab import files
import zipfile
import os
import shutil

# ============================================================
# 1. CLEAN OLD PROJECT DIRECTORIES
# ============================================================

for path in [
    "/content/github-mcp-server",
    "/content/github-mcp-server-clean",
    "/content/mcp_upload"
]:
    if os.path.exists(path):
        shutil.rmtree(path)

os.makedirs("/content/mcp_upload", exist_ok=True)

# ============================================================
# 2. UPLOAD ZIP
# ============================================================

print("📦 Select your github-mcp-server.zip file...")

uploaded = files.upload()

if not uploaded:
    raise RuntimeError("❌ No file was uploaded.")

zip_filename = next(iter(uploaded.keys()))

if not zip_filename.lower().endswith(".zip"):
    raise RuntimeError(f"❌ Expected a ZIP file, got: {zip_filename}")

print(f"\n✅ Uploaded: {zip_filename}")

# ============================================================
# 3. EXTRACT ZIP TO TEMPORARY LOCATION
# ============================================================

temp_extract = "/content/mcp_upload/extracted"
os.makedirs(temp_extract, exist_ok=True)

with zipfile.ZipFile(zip_filename, "r") as zip_ref:
    zip_ref.extractall(temp_extract)

print("✅ ZIP extracted.")

# ============================================================
# 4. FIND ACTUAL PROJECT ROOT
#    We look for server.py + client_demo.py
# ============================================================

project_root = None

for root, dirs, filenames in os.walk(temp_extract):
    if "server.py" in filenames and "client_demo.py" in filenames:
        project_root = root
        break

if project_root is None:
    print("\n❌ Could not find the project files.")
    print("Expected:")
    print("  server.py")
    print("  client_demo.py")
    print("  requirements.txt")
    raise RuntimeError("Project root could not be detected.")

print(f"\n🔎 Detected project root:")
print(project_root)

# ============================================================
# 5. CREATE ONE CLEAN PROJECT DIRECTORY
# ============================================================

clean_project = "/content/github-mcp-server"

os.makedirs(clean_project, exist_ok=True)

# Copy project contents, NOT the parent folder
for item in os.listdir(project_root):
    source = os.path.join(project_root, item)
    destination = os.path.join(clean_project, item)

    if os.path.isdir(source):
        shutil.copytree(source, destination, dirs_exist_ok=True)
    else:
        shutil.copy2(source, destination)

# ============================================================
# 6. REMOVE PYTHON CACHE / PYTEST CACHE
# ============================================================

for root, dirs, filenames in os.walk(clean_project):
    for directory in list(dirs):
        if directory in {"__pycache__", ".pytest_cache"}:
            shutil.rmtree(os.path.join(root, directory))

# ============================================================
# 7. VERIFY FINAL STRUCTURE
# ============================================================

print("\n" + "=" * 60)
print("✅ CLEAN PROJECT CREATED")
print("=" * 60)

print(f"\n📁 Project location:")
print(clean_project)

print("\n📂 Final structure:")

for root, dirs, filenames in os.walk(clean_project):
    level = root.replace(clean_project, "").count(os.sep)
    indent = "    " * level

    print(f"{indent}{os.path.basename(root)}/")

    for filename in sorted(filenames):
        print(f"{indent}    └── {filename}")

print("\n" + "=" * 60)
print("🎉 Upload + extraction completed successfully!")
print("=" * 60)

📦 Select your github-mcp-server.zip file...


Saving github-mcp-server.zip to github-mcp-server.zip

✅ Uploaded: github-mcp-server.zip
✅ ZIP extracted.

🔎 Detected project root:
/content/mcp_upload/extracted/github-mcp-server

✅ CLEAN PROJECT CREATED

📁 Project location:
/content/github-mcp-server

📂 Final structure:
github-mcp-server/
    └── .env.example
    └── .gitignore
    └── README.md
    └── client_demo.py
    └── requirements.txt
    └── server.py
    tests/
        └── test_server.py

🎉 Upload + extraction completed successfully!


In [2]:
%cd /content/github-mcp-server

import os
import sys

sys.path.insert(0, os.getcwd())

print("📍 Current directory:")
print(os.getcwd())

print("\n📁 Project files:")

for root, dirs, files_list in os.walk("."):
    level = root.count(os.sep)
    indent = "    " * level

    print(f"{indent}{os.path.basename(root)}/")

    for filename in sorted(files_list):
        print(f"{indent}    └── {filename}")

/content/github-mcp-server
📍 Current directory:
/content/github-mcp-server

📁 Project files:
./
    └── .env.example
    └── .gitignore
    └── README.md
    └── client_demo.py
    └── requirements.txt
    └── server.py
    tests/
        └── test_server.py


In [3]:
!pip install -q "mcp[cli]" requests pytest

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 350.0/350.0 kB 10.5 MB/s eta 0:00:00


In [4]:
import mcp
import requests
import pytest

print("✅ Dependencies imported successfully")

✅ Dependencies imported successfully


In [5]:
!python -m pytest -q

................                                                         [100%]
16 passed in 2.08s


In [6]:
!python client_demo.py

GitHub MCP Server - Client Demo
Target repository for the demo: modelcontextprotocol/python-sdk

A. Tool discovery
  - search_repositories: Search GitHub repositories using the GitHub Search API.

Args:
    query: GitHub search query, e.g. "modelcontextprotocol language:python".
    limit: Maximum number of repositories to return (1-20).

  - get_repository: Get metadata about a single GitHub repository.

Args:
    owner: Repository owner (user or organization login).
    repo: Repository name.

  - list_issues: List issues for a GitHub repository.

Args:
    owner: Repository owner (user or organization login).
    repo: Repository name.
    state: One of "open", "closed", "all".
    limit: Maximum number of issues to return (1-20).


B. search_repositories
  Query: modelcontextprotocol language:python  (showing 5 results)
  - tadata-org/fastapi_mcp (11983 stars, Python)
    https://github.com/tadata-org/fastapi_mcp
  - mrexodia/ida-pro-mcp (11505 stars, Python)
    https://github.com